# Petroleum Supply Analysis — Phase 2
## 1. Verify Raw Data Access
Check that the historical and incremental raw files are available.

In [0]:
from pathlib import Path

# Location of the validated Phase 1 raw data.
raw_base_path = Path(
    "/Volumes/workspace/default/petroleum_raw/phase1_expanded/"
    "20261003T114452938244Z"
)

if not raw_base_path.is_dir():
    raise FileNotFoundError(f"Raw data directory not found: {raw_base_path}")

for load_type in ("full_load", "incremental_load"):
    load_path = raw_base_path / load_type
    json_files = sorted(load_path.glob("page_*.json"))

    if not json_files:
        raise FileNotFoundError(f"No raw JSON files found in: {load_path}")

    total_size_mb = sum(
        file.stat().st_size for file in json_files
    ) / 1_000_000

    print(
        f"{load_type}: "
        f"{len(json_files)} files | "
        f"{total_size_mb:.2f} MB"
    )

print("Raw data access verified.")

## 2. Configure Pipeline Parameters
Select the raw data batch and load type. These parameters allow
historical batches to be reprocessed without changing the pipeline code.

In [0]:
from pathlib import Path

# Default values point to the validated Phase 1 batch.
dbutils.widgets.text(
    "raw_root",
    "/Volumes/workspace/default/petroleum_raw/phase1_expanded",
    "Raw Data Root"
)

dbutils.widgets.text(
    "batch_id",
    "20261003T114452938244Z",
    "Batch ID"
)

dbutils.widgets.dropdown(
    "load_type",
    "full_load",
    ["full_load", "incremental_load"],
    "Load Type"
)

raw_root = dbutils.widgets.get("raw_root").strip()
batch_id = dbutils.widgets.get("batch_id").strip()
load_type = dbutils.widgets.get("load_type").strip()

if not raw_root.startswith("/Volumes/"):
    raise ValueError("Raw Data Root must be a /Volumes/ path.")

if not batch_id or "/" in batch_id or batch_id in {".", ".."}:
    raise ValueError("Batch ID must be a single directory name.")

if load_type not in {"full_load", "incremental_load"}:
    raise ValueError("Unsupported load type.")

source_path = Path(raw_root) / batch_id / load_type
source_files = sorted(source_path.glob("page_*.json"))

if not source_files:
    raise FileNotFoundError(f"No raw JSON files found in: {source_path}")

print(f"Batch ID: {batch_id}")
print(f"Load type: {load_type}")
print(f"Source path: {source_path}")
print(f"Files available: {len(source_files)}")